# CineMatch: Hybrid Movie Recommendation System
### Classical Machine Learning (TF-IDF) + Deep Learning (Sentence Transformer Embeddings)

This notebook presents the complete architecture, data engineering, model development, hybrid synthesis, and quantitative evaluation for the **CineMatch** movie recommendation engine.

---
### System Architecture
1. **Data Ingestion & Cleaning**: Ingestion of MovieLens 25M (`movies.csv`, `ratings.csv`, and TMDB mapping via `links.csv`).
2. **Feature Engineering**: Combining cleaned movie titles, franchise identifiers, weighted multi-genres, release eras/decades, and Bayesian community rating signals into a unified metadata *soup*.
3. **Classical ML (TF-IDF)**: Sublinear unigram/bigram document-term matrix with pairwise cosine similarity.
4. **Deep Learning (Sentence-BERT)**: Pretrained `all-MiniLM-L6-v2` dense transformer embeddings (384-dimensional) capturing semantic proximity.
5. **Hybrid Recommendation Engine**: Optimal weighted linear combination $S_{\text{Hybrid}} = \alpha S_{\text{ML}} + (1 - \alpha) S_{\text{DL}}$.
6. **Rigorous Validation**: Quantitative evaluation using Precision@K, Recall@K, Hit Rate@K, Jaccard Similarity, and Intra-List Diversity (ILD).


In [ ]:
import os
import re
import json
import time
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("All required libraries loaded successfully.")

## 1. Dataset Ingestion & Preprocessing
We load `movies.csv` and `data/links.csv` (for TMDB poster integration), and aggregate ratings from `ratings.csv` to calculate community popularity and Bayesian mean scores.

In [ ]:
# Paths
movies_path = "../movies.csv" if os.path.exists("../movies.csv") else "movies.csv"
ratings_path = "../ratings.csv" if os.path.exists("../ratings.csv") else "ratings.csv"
links_path = "../data/links.csv" if os.path.exists("../data/links.csv") else "data/links.csv"

movies_df = pd.read_csv(movies_path)
links_df = pd.read_csv(links_path) if os.path.exists(links_path) else pd.DataFrame(columns=["movieId", "tmdbId"])

print(f"Total raw movies: {len(movies_df):,}")

# Aggregating 25M ratings efficiently in chunks
chunks = []
for chunk in pd.read_csv(ratings_path, usecols=["movieId", "rating"], chunksize=5000000):
    agg = chunk.groupby("movieId").agg(count=("rating", "count"), sum=("rating", "sum")).reset_index()
    chunks.append(agg)

all_agg = pd.concat(chunks).groupby("movieId").sum().reset_index()
all_agg["rating_mean"] = (all_agg["sum"] / all_agg["count"]).round(2)

movies_df = movies_df.merge(all_agg[["movieId", "count", "rating_mean"]], on="movieId", how="left")
movies_df["rating_count"] = movies_df["count"].fillna(0).astype(int)
movies_df["rating_mean"] = movies_df["rating_mean"].fillna(0.0)

# Merge TMDB IDs
movies_df = movies_df.merge(links_df[["movieId", "tmdbId"]], on="movieId", how="left")
movies_df["movie_id"] = movies_df["tmdbId"].fillna(movies_df["movieId"]).astype(int)

# Filter catalog: valid genres, top 6000 by popularity + rating quality
cleaned_df = movies_df[movies_df["genres"].notna() & (movies_df["genres"] != "(no genres listed)")].copy()
cleaned_df = cleaned_df.sort_values(by=["rating_count", "rating_mean"], ascending=[False, False])
catalog_df = cleaned_df.head(6000).copy().reset_index(drop=True)

print(f"Indexed top catalog: {len(catalog_df):,} movies (min ratings: {catalog_df['rating_count'].min()})")
catalog_df[["movieId", "movie_id", "title", "genres", "rating_count", "rating_mean"]].head(10)

## 2. Feature Engineering: Metadata Content Soup
We synthesize a rich composite feature *soup* combining:
- Normalized title tokens (e.g. `The Matrix`, `The Godfather`)
- Extracted franchise and series identifiers (e.g. `franchise_toy_story`, `franchise_harry_potter`)
- Weighted primary & secondary genres + bi-gram genre combinations
- Release decade and release year tokens
- Audience sentiment and critical acclaim profile tags (`critically_acclaimed`, `mega_blockbuster`)

In [ ]:
def format_display_title(title):
    m = re.match(r"^(.*?),\s*(The|A|An)\s*(\(\d{4}\))?$", str(title).strip())
    if m:
        prefix = m.group(2)
        main = m.group(1)
        year = m.group(3) or ""
        return f"{prefix} {main} {year}".strip()
    return str(title).strip()

def get_clean_title_name(title):
    t = re.sub(r"\s*\(\d{4}\)", "", str(title))
    m = re.match(r"^(.*?),\s*(The|A|An)$", t.strip())
    if m:
        t = f"{m.group(2)} {m.group(1)}"
    return t.strip()

def extract_year(title):
    m = re.search(r"\((\d{4})\)", str(title))
    return int(m.group(1)) if m else 2000

def extract_franchise(clean_name):
    parts = re.split(r"[:\-–]", clean_name)
    if len(parts) > 1 and len(parts[0].strip()) > 2:
        return "franchise_" + re.sub(r"[^a-zA-Z0-9]", "_", parts[0].strip().lower())
    return ""

catalog_df["display_title"] = catalog_df["title"].apply(format_display_title)
catalog_df["clean_name"] = catalog_df["title"].apply(get_clean_title_name)
catalog_df["year"] = catalog_df["title"].apply(extract_year)
catalog_df["decade"] = (catalog_df["year"] // 10 * 10).astype(str) + "s"
catalog_df["franchise"] = catalog_df["clean_name"].apply(extract_franchise)

feature_soups = []
for _, row in catalog_df.iterrows():
    genres_list = [g.strip() for g in row["genres"].split("|") if g != "(no genres listed)"]
    genre_str = " ".join(genres_list).lower()
    genre_pairs = []
    for i in range(len(genres_list)):
        for j in range(i + 1, len(genres_list)):
            genre_pairs.append(f"{genres_list[i].lower()}_{genres_list[j].lower()}")
    genre_pairs_str = " ".join(genre_pairs)

    clean_name_tokens = re.sub(r"[^a-zA-Z0-9\s]", " ", row["clean_name"]).lower()
    franchise_token = row["franchise"]
    decade_token = f"decade_{row['decade']}"
    year_token = f"year_{row['year']}"

    rating_tags = []
    if row.get("rating_mean", 0) >= 4.0:
        rating_tags.append("critically_acclaimed masterpiece top_rated")
    elif row.get("rating_mean", 0) >= 3.6:
        rating_tags.append("highly_rated popular_favorite")

    if row.get("rating_count", 0) >= 5000:
        rating_tags.append("mega_blockbuster timeless_classic")
    elif row.get("rating_count", 0) >= 1000:
        rating_tags.append("widely_watched blockbuster")

    audience_str = " ".join(rating_tags)
    soup = f"{clean_name_tokens} {clean_name_tokens} {franchise_token} {genre_str} {genre_str} {genre_pairs_str} {decade_token} {year_token} {audience_str}"
    feature_soups.append(soup)

catalog_df["soup"] = feature_soups
print("Sample soup:")
print(catalog_df["soup"].iloc[0])

## 3. Classical Machine Learning: TF-IDF Representation & Cosine Similarity
Using `scikit-learn`'s `TfidfVectorizer`, we construct a sublinear unigram/bigram document-term matrix and calculate pairwise cosine similarity.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(catalog_df["soup"])
ml_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix).astype(np.float32)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"ML similarity matrix shape: {ml_similarity.shape}")

## 4. Deep Learning Component: Pretrained Sentence Transformer Embeddings
We utilize the `all-MiniLM-L6-v2` transformer model to produce 384-dimensional dense semantic vectors. This captures thematic relationships and conceptual nuances that lexical token matching misses.

In [ ]:
dl_model = SentenceTransformer("all-MiniLM-L6-v2")
dl_embeddings = dl_model.encode(
    catalog_df["soup"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)
dl_similarity = np.dot(dl_embeddings, dl_embeddings.T).astype(np.float32)

print(f"DL Embeddings shape: {dl_embeddings.shape}")
print(f"DL Similarity Matrix shape: {dl_similarity.shape}")

## 5. Hybrid Recommendation Synthesis
The hybrid similarity combines lexical exactness with semantic generalization:
$$S_{\text{Hybrid}}(i, j) = \alpha S_{\text{ML}}(i, j) + (1 - \alpha) S_{\text{DL}}(i, j)$$
with $\alpha = 0.50$.

In [ ]:
alpha = 0.50
hybrid_similarity = (alpha * ml_similarity + (1.0 - alpha) * dl_similarity).astype(np.float32)
print(f"Constructed Hybrid similarity matrix of shape: {hybrid_similarity.shape}")

## 6. Model Evaluation & Quantitative Benchmarks
We evaluate Precision@5, Recall@5, Hit Rate@5, Jaccard Index, and Intra-List Diversity (ILD) over 500 query movies.

In [ ]:
def evaluate_model(sim_matrix, catalog_df, k=5, num_queries=500):
    np.random.seed(42)
    sample_indices = np.random.choice(len(catalog_df), size=min(num_queries, len(catalog_df)), replace=False)
    genre_sets = [set(g.split("|")) for g in catalog_df["genres"]]

    precisions, recalls, hit_rates, jaccards, ild_scores = [], [], [], [], []

    for idx in sample_indices:
        q_genres = genre_sets[idx]
        if not q_genres:
            continue

        scores = sim_matrix[idx]
        top_k = np.argsort(scores)[::-1][1:k+1]

        k_precs = []
        recovered = set()
        hit = 0
        k_jacc = []

        for r_idx in top_k:
            r_g = genre_sets[r_idx]
            inter = q_genres.intersection(r_g)
            union = q_genres.union(r_g)
            k_precs.append(len(inter) / len(r_g) if r_g else 0.0)
            if inter:
                hit += 1
            if union:
                k_jacc.append(len(inter) / len(union))
            recovered.update(inter)

        sub_matrix = sim_matrix[np.ix_(top_k, top_k)]
        off_diag = sub_matrix[~np.eye(k, dtype=bool)]
        ild = 1.0 - float(np.mean(off_diag)) if len(off_diag) > 0 else 0.0

        precisions.append(np.mean(k_precs) if k_precs else 0.0)
        recalls.append(len(recovered) / len(q_genres) if q_genres else 0.0)
        hit_rates.append(1.0 if hit >= 1 else 0.0)
        jaccards.append(np.mean(k_jacc) if k_jacc else 0.0)
        ild_scores.append(ild)

    return {
        "Precision@5": np.mean(precisions),
        "Recall@5": np.mean(recalls),
        "Hit Rate@5": np.mean(hit_rates),
        "Jaccard@5": np.mean(jaccards),
        "Diversity (ILD@5)": np.mean(ild_scores)
    }

results = {
    "TF-IDF (Classical ML)": evaluate_model(ml_similarity, catalog_df),
    "Sentence-BERT (DL)": evaluate_model(dl_similarity, catalog_df),
    "Hybrid Recommender": evaluate_model(hybrid_similarity, catalog_df)
}

df_eval = pd.DataFrame(results).T
df_eval

## 7. Interactive Recommendation Verification
Testing recommendation outputs for notable benchmark films.

In [ ]:
def get_recommendations(movie_title, k=5):
    matches = catalog_df[catalog_df["title"] == movie_title]
    if len(matches) == 0:
        matches = catalog_df[catalog_df["display_title"].str.contains(movie_title, case=False, na=False)]
    if len(matches) == 0:
        print(f"Movie '{movie_title}' not found.")
        return

    idx = matches.index[0]
    matched_title = catalog_df.iloc[idx]["title"]
    matched_genres = catalog_df.iloc[idx]["genres"]
    print(f"\nQuery Movie: {matched_title} (Genres: {matched_genres})")
    print("-" * 75)

    top_indices = np.argsort(hybrid_similarity[idx])[::-1][1:k+1]
    for rank, rec_idx in enumerate(top_indices, 1):
        rec_title = catalog_df.iloc[rec_idx]["title"]
        rec_genres = catalog_df.iloc[rec_idx]["genres"]
        score = hybrid_similarity[idx, rec_idx]
        print(f"  #{rank}: {rec_title:<45} | {rec_genres:<32} | Score: {score:.4f}")

# Run queries
get_recommendations("Toy Story (1995)")
get_recommendations("Matrix, The (1999)")
get_recommendations("Godfather, The (1972)")
get_recommendations("Dark Knight, The (2008)")
get_recommendations("Interstellar (2014)")